# m12 lab · The surface ECG and the forward problem

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

The surface ECG is the heart's current sources seen through the torso. You compute the potential of a dipole in a
spherical torso, build the limb leads and check Einthoven's law, read the frontal axis, generate a QRS-like signal from
a front crossing the heart, and see how an off-center heart bends the lead vectors.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A dipole in a spherical torso | eq-m12-sphere, eq-m12-series |
| B | Lead vectors, Einthoven's law and the augmented leads | eq-m12-lead, eq-m12-einthoven, eq-m12-augmented |
| C | The frontal axis, with noise | eq-m12-axis |
| D | A QRS from a front crossing the heart | eq-m12-forward |
| E | An off-center heart | eq-m12-series, eq-m12-lead |

The torso is a homogeneous, insulated sphere of radius 1 and conductivity 1 (relative units). Each part ends with
**checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import ecg as G

## Part A · A dipole in a spherical torso

In an infinite medium a dipole's potential at distance $R$ is $p\cos\theta/(4\pi\sigma R^2)$ (m01). Inside an insulated
sphere, current cannot leave, and the surface potential triples (eq-m12-sphere). For an off-center source the surface
potential is a Legendre series (eq-m12-series). Compare the two near the center, and the surface map.

In [ ]:
p = np.array([0.3, 0.8, 0.2])
theta = np.linspace(0, 2 * np.pi, 181)
ring = np.stack([np.cos(theta), np.sin(theta), np.zeros_like(theta)], axis=1)
closed = np.array([G.centric_surface(p, x) for x in ring])
series = np.array([G.dipole_surface(p, [0.0, 0.0, 1e-3], x) for x in ring])
infinite = ring @ p / (4 * np.pi)
print(f"largest difference closed form vs series: {np.max(np.abs(closed - series)):.1e}")
print(f"surface / infinite-medium ratio: {np.max(np.abs(closed)) / np.max(np.abs(infinite)):.3f}")
plt.plot(np.degrees(theta), closed, label="sphere surface"); plt.plot(np.degrees(theta), infinite, "--", label="infinite medium")
plt.xlabel("angle in the frontal plane (degrees)"); plt.ylabel("potential (relative)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert np.max(np.abs(closed - series)) < 2e-4
assert abs(np.max(np.abs(closed)) / np.max(np.abs(infinite)) - 3.0) < 1e-9

## Part B · Lead vectors, Einthoven's law and the augmented leads

Every lead is a weighted difference of electrode potentials, so for one dipole its voltage is $\mathbf L\cdot\mathbf p$
(eq-m12-lead), the idea Burger and van Milaan called the lead vector (1947). The three limb leads share electrodes, so
$\text{I} + \text{III} = \text{II}$ whatever the heart does (eq-m12-einthoven); the augmented leads compare each limb
with the average of the other two (eq-m12-augmented).

In [ ]:
lv = {k: G.lead_vector(G.LIMBS[a], G.LIMBS[b]) for k, (a, b) in {"I": ("L", "R"), "II": ("F", "R"), "III": ("F", "L")}.items()}
for k, v in lv.items():
    print(f"lead {k:3s} vector {np.round(v, 4)}, frontal angle {np.degrees(np.arctan2(v[1], v[0])):6.1f} degrees")
rng = np.random.default_rng(1)
resid = []
for _ in range(100):
    L = G.leads_for_dipole(rng.standard_normal(3))
    resid.append(L["I"] + L["III"] - L["II"])
    resid.append(L["aVR"] + L["aVL"] + L["aVF"])
print(f"largest |I + III - II| and |aVR + aVL + aVF| over 100 random dipoles: {np.max(np.abs(resid)):.1e}")

In [ ]:
# Checks
angles = [np.degrees(np.arctan2(v[1], v[0])) for v in lv.values()]
assert np.allclose(angles, [0.0, 60.0, 120.0])
assert np.max(np.abs(resid)) < 1e-12

## Part C · The frontal axis

With lead vectors at 0° (I) and 90° (aVF, $\sqrt3/2$ as long), the frontal angle of the heart vector is
$\operatorname{atan2}(2\,\text{aVF}/\sqrt3, \text{I})$ (eq-m12-axis). It is exact in this model. Add noise to the two
leads and see how the estimate spreads when the vector is small.

In [ ]:
errs = {}
for mag in (1.0, 0.2):
    e = []
    for ang in np.radians(np.arange(-90, 181, 15)):
        pv = mag * np.array([np.cos(ang), np.sin(ang), 0.0])
        L = G.leads_for_dipole(pv)
        for _ in range(20):
            i = L["I"] + 0.01 * rng.standard_normal()
            f = L["aVF"] + 0.01 * rng.standard_normal()
            d = G.frontal_axis(i, f) - np.degrees(ang)
            e.append((d + 180) % 360 - 180)
    errs[mag] = np.std(e)
    print(f"heart vector magnitude {mag}: axis error SD {errs[mag]:.1f} degrees (noise 0.01 per lead)")

In [ ]:
# Checks
assert errs[0.2] > 3 * errs[1.0]

## Part D · A QRS from a front crossing the heart

A planar activation front carries a dipole moment proportional to its area (m01). Sweep one through a ball of radius
30 mm at 0.6 m/s (illustrative values) and the heart vector rises and falls along the front's direction
(eq-m12-forward). Project it on the lead vectors to get limb-lead signals, and read the axis from the areas of I and
aVF.

In [ ]:
t = np.linspace(0, 120, 1201)
direction = np.array([np.cos(np.radians(50)), np.sin(np.radians(50)), 0.3])
pt = G.front_heart_vector(t, direction, radius=30.0, cv=0.6) * 1e-4
sig = G.lead_signals(pt, lv)
sig["aVF"] = (sig["II"] + sig["III"]) / 2
dur = t[np.linalg.norm(pt, axis=1) > 0][[0, -1]]
axis_qrs = G.frontal_axis(np.trapezoid(sig["I"], t), np.trapezoid(sig["aVF"], t))
print(f"QRS-like duration {dur[1] - dur[0]:.1f} ms (2a / CV = {2 * 30 / 0.6:.0f} ms); axis from areas {axis_qrs:.1f} degrees")
for k in ("I", "II", "III", "aVF"):
    plt.plot(t, sig[k], label=k)
plt.xlabel("time (ms)"); plt.ylabel("lead voltage (relative)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert abs((dur[1] - dur[0]) - 100.0) < 0.2
assert abs(axis_qrs - 50.0) < 1e-6

## Part E · An off-center heart

The heart sits left of and in front of the torso's center. Move the dipole 0.3 of the radius to the left and the
limb-lead vectors no longer form an equilateral triangle; the textbook axis formula then misreads the heart vector.
Burger and van Milaan's lead-vector work and Rudy's eccentric-spheres model explore this kind of effect.

In [ ]:
for off in ([0.0, 0, 0], [0.3, 0, 0], [0.3, 0.1, 0.2]):
    vecs = {k: G.lead_vector(G.LIMBS[a], G.LIMBS[b], off) for k, (a, b) in {"I": ("L", "R"), "II": ("F", "R"), "III": ("F", "L")}.items()}
    ang = {k: np.degrees(np.arctan2(v[1], v[0])) for k, v in vecs.items()}
    L = G.leads_for_dipole(np.array([np.cos(np.radians(60)), np.sin(np.radians(60)), 0.0]), r0=off)
    print(f"heart at {off}: lead angles " + ", ".join(f"{k} {a:.0f}" for k, a in ang.items())
          + f"; axis formula for a 60-degree vector: {G.frontal_axis(L['I'], L['aVF']):.0f}")
err_off = abs(G.frontal_axis(*[G.leads_for_dipole(np.array([1.0, 0, 0]), r0=[0.3, 0, 0])[k] for k in ("I", "aVF")]))

In [ ]:
# Checks
assert err_off > 5

## Where to go next

- Add precordial electrodes on the anterior sphere and compute their lead vectors relative to the Wilson central
  terminal.
- Replace the sphere with a 2D torso cross-section containing low-conductivity lungs, using the m02 solver, and see how
  the lead vectors change.
- m32 inverts this forward model to image the heart's potentials; m37 learns from real 12-lead ECGs.